In [16]:
import deepmimo as dm
import numpy as np
from utils.utils import ChannelParameters, compute_single_array_response_torch
import torch

In [44]:
SPEED = 3 #km/h
TIME_STEP = 1 #ms
DURATION = 24 #timesteps

DISTANCE_STEPS = (SPEED * 1e3) * (TIME_STEP * 1e-6)
DISTANCE_STEPS

0.003

{'bs_antenna': {'shape': array([8, 1]),
  'spacing': 0.5,
  'rotation': array([   0,    0, -135]),
  'radiation_pattern': 'isotropic'},
 'ue_antenna': {'shape': array([1, 1]),
  'spacing': 0.5,
  'rotation': array([0, 0, 0]),
  'radiation_pattern': 'isotropic'},
 'doppler': 0,
 'num_paths': 25,
 'freq_domain': 1,
 'ofdm': {'subcarriers': 32,
  'selected_subcarriers': array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
         17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31]),
  'bandwidth': 960000.0,
  'rx_filter': 0}}

In [ ]:
scenario = 'city_0_newyork_3p5_s'
dataset = dm.load(scenario)
dataset.compute_channels(ChannelParameters())
channels = dataset.channels

Loading TXRX PAIR: TXset 0 (tx_idx 0) & RXset 1 (rx_idxs 32123)
Loading TXRX PAIR: TXset 0 (tx_idx 1) & RXset 1 (rx_idxs 32123)
Loading TXRX PAIR: TXset 0 (tx_idx 2) & RXset 1 (rx_idxs 32123)
No doppler in channel generation because all velocities are zero


Generating channels: 100%|██████████| 32123/32123 [00:01<00:00, 18062.28it/s]


No doppler in channel generation because all velocities are zero


Generating channels: 100%|██████████| 32123/32123 [00:01<00:00, 17792.15it/s]


No doppler in channel generation because all velocities are zero


Generating channels: 100%|██████████| 32123/32123 [00:01<00:00, 18047.54it/s]


In [73]:
def squeeze_channel_to_matrix(channel):
    channel = np.asarray(channel).squeeze()
    if channel.ndim != 2:
        raise ValueError(f"Expected a 2D per-user channel after squeeze, got shape {channel.shape}")
    return channel.astype(np.complex64, copy=False)
def compute_beam_label_from_channel(H, S):
    if not torch.is_tensor(H):
        H = torch.from_numpy(H)
    if not torch.is_tensor(S):
        S = torch.from_numpy(S)
    H = H.to(torch.complex64)
    S = S.to(torch.complex64)
    Y = S.conj().T @ H
    prx = torch.sum(torch.abs(Y) ** 2, dim=2)
    best = torch.argmax(prx, dim=1)
    return best, prx

def get_direction_candidates(rx_pos, user_pos, direction, duration):
    x_vals = rx_pos[:, 0]
    y_vals = rx_pos[:, 1]
    user_x = user_pos[0]
    user_y = user_pos[1]

    if direction == "left":
        mask = np.isclose(y_vals, user_y) & (x_vals < user_x)
        candidates = np.where(mask)[0]
        #x_vals:  
        order = np.argsort(x_vals[candidates])
        ordered = candidates[order]
        
        # print(f"left {ordered[-duration:]} ")

        return ordered[-duration:]
    if direction == "right":
        mask = np.isclose(y_vals, user_y) & (x_vals > user_x)
        candidates = np.where(mask)[0]
        order = np.argsort(-x_vals[candidates])
        ordered = candidates[order]
        return ordered[:duration]
    if direction == "down":
        mask = np.isclose(x_vals, user_x) & (y_vals < user_y)
        candidates = np.where(mask)[0]
        order = np.argsort(y_vals[candidates])
        ordered = candidates[order]
        # print(f"down {ordered[-duration:]} ")
        return ordered[-duration:]
    if direction == "up":
        mask = np.isclose(x_vals, user_x) & (y_vals > user_y)
        candidates = np.where(mask)[0]
        order = np.argsort(-y_vals[candidates])
        ordered = candidates[order]
        return ordered[:duration]
    raise ValueError(f"Unknown direction: {direction}")


def pick_history_indices(rx_pos, user_idx, duration):
    user_pos = rx_pos[user_idx]
    directions = ["left", "right", "down", "up"]

    best_direction = None
    best_candidates = np.array([], dtype=np.int64)
    for direction in directions:
        candidates = get_direction_candidates(rx_pos, user_pos, direction, duration)
        if candidates.size >= duration:
            return candidates[-duration:] if direction in {"left", "down"} else candidates[:duration], direction
        if candidates.size > best_candidates.size:
            best_candidates = candidates
            best_direction = direction

    if best_candidates.size == 0:
        history = np.full(duration, user_idx, dtype=np.int64)
        return history, "self"

    if best_direction in {"left", "down"}:
        trimmed = best_candidates
        pad_value = trimmed[0]
    else:
        trimmed = best_candidates
        pad_value = trimmed[0]

    if trimmed.size < duration:
        pad = np.full(duration - trimmed.size, pad_value, dtype=np.int64)
        history = np.concatenate([pad, trimmed]).astype(np.int64, copy=False)
    else:
        history = trimmed[-duration:].astype(np.int64, copy=False)
    return history, best_direction


def build_movement_sequences(channel_tx, rx_pos, indices, duration):
    '''
    indices: train or test user indices
    duration: steps

    '''
    split_channels = np.asarray(channel_tx)[indices]
    split_rx_pos = np.asarray(rx_pos)[indices] #all the train or positions

    sequence_channels = []
    directions = []
    
    for local_idx in range(len(indices)):
        history_indices, direction = pick_history_indices(split_rx_pos, local_idx, duration)
        ordered_indices = list(history_indices) + [local_idx]
        matrices = [squeeze_channel_to_matrix(split_channels[idx]) for idx in ordered_indices]
        # print(f"len(matrices) : {len(matrices)} {matrices[0].shape}")
        sequence = np.array(matrices)
        # sequence = np.concatenate(matrices, axis=-1)

        # print(sequence.shape, np.array(matrices).shape)
        sequence_channels.append(sequence.astype(np.complex64, copy=False))
        directions.append(direction)

    return np.stack(sequence_channels, axis=0), directions


def make_dft_codebook(B=8):
    params = ChannelParameters()
    az_t = np.linspace(-np.pi, np.pi, B, endpoint=False, dtype=np.float32)
    el_t = np.linspace(-np.pi, np.pi, B, endpoint=False, dtype=np.float32)
    az_new = []
    el_new = []
    for az in az_t:
        for el in el_t:
            az_new.append(az)
            el_new.append(el)
    az_new = torch.tensor(az_new).unsqueeze(1)
    el_new = torch.tensor(el_new).unsqueeze(1)
    array_response = compute_single_array_response_torch(params.bs_antenna, az_new, el_new)
    return array_response.squeeze(2).T

In [74]:
seed= 42
train_ratio = 0.8
if hasattr(dataset, "n_ue") and isinstance(dataset.n_ue, int):
    dataset = [dataset]
    channels = [channels]
duration = DURATION
train_channels = []
train_labels = []
test_channels = []
test_labels = []
S = make_dft_codebook()
for tx in range(len(dataset)):
    channel_tx = np.asarray(channels[tx])#[n_users, n_rx_ant, n_tx_ant, n_subcarriers]
    n_ue = dataset[tx].n_ue
    indices = np.arange(n_ue)
    np.random.seed(seed + tx)
    np.random.shuffle(indices)
    split_idx = int(train_ratio * len(indices))
    train_indices = indices[:split_idx]
    test_indices = indices[split_idx:]

    use_indices = dataset[tx].los != -1
    train_indices = np.array([i for i in train_indices if use_indices[i]], dtype=np.int64)
    test_indices = np.array([i for i in test_indices if use_indices[i]], dtype=np.int64)

    labels, _ = compute_beam_label_from_channel(channel_tx.squeeze(1), S)
    
    train_channel_seq, train_dirs = build_movement_sequences(
        channel_tx * 1e6, dataset[tx].rx_pos, train_indices, duration
    )
    test_channel_seq, test_dirs = build_movement_sequences(
        channel_tx * 1e6, dataset[tx].rx_pos, test_indices, duration
    )

    print(f" channel_tx: {train_channel_seq.shape}")
    print(
        f"TX {tx}: train={len(train_indices)} val={len(test_indices)} "
        f"train_dirs={dict((d, train_dirs.count(d)) for d in sorted(set(train_dirs)))} "
        f"val_dirs={dict((d, test_dirs.count(d)) for d in sorted(set(test_dirs)))}"
    )

    train_channels.append(train_channel_seq)
    train_labels.append(labels[train_indices].cpu().numpy())
    test_channels.append(test_channel_seq)
    test_labels.append(labels[test_indices].cpu().numpy())
    break

 channel_tx: (18282, 25, 8, 32)
TX 0: train=18282 val=4531 train_dirs={'left': 15639, 'right': 2640, 'self': 3} val_dirs={'down': 12, 'left': 2296, 'right': 2183, 'up': 40}


In [75]:
train_channel_seq[0,0,0,0]


np.complex64(1.5253328-1.1749231j)

In [76]:
train_channel_seq[0,0,0,-1]

np.complex64(-1.5479394-1.14033j)